# Cleaning 03: Quarantine Labels and Export

Use this notebook to remove non-trainable rows, validate the final dataset, and write the export files.

In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from helpers.logger import log_step, save_log
from helpers.data_loader import DataLoader
from helpers.profile import profile

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

STAGE2_PATH = DataLoader.interim("cleaning/cleaning_stage2.csv")
PROCESSED_PATH = DataLoader.processed("merged_inspections_licenses_inner_clean.csv")
PROCESSED_PARQUET_PATH = DataLoader.processed("merged_inspections_licenses_inner_clean.parquet")
LOG_PATH = DataLoader.interim("cleaning/cleaning_log.csv")
QUARANTINE_PATH = DataLoader.interim("cleaning/quarantine.csv")
FINAL_LOG_PATH = DataLoader.interim("cleaning/cleaning_final_log.csv")

df_clean = pd.read_csv(STAGE2_PATH)
print('Loaded stage 2 shape:', df_clean.shape)

Loaded stage 2 shape: (196676, 26)


In [ ]:
profile(df_clean, 'Stage 2 input profile').head(20)

=== Stage 2 input profile ===
shape: (196676, 26)
full duplicates: 0
duplicate Inspection ID: 0


,missing_count,missing_pct,dtype
LICENSE TERM START DATE,155546,79.09,object
Violations,52234,26.56,object
COMMUNITY AREA NAME,11558,5.88,object
LICENSE TERM EXPIRATION DATE,9706,4.94,object
APPLICATION TYPE,9690,4.93,object
LICENSE DESCRIPTION,9690,4.93,object
LICENSE STATUS,9690,4.93,object
Facility Type,4768,2.42,object
AKA Name,2457,1.25,object
Latitude,689,0.35,float64


In [3]:
df_clean['Results'].value_counts()

Results
Pass                    106003
Fail                     38055
Pass w/ Conditions       27412
Out of Business          16909
No Entry                  6322
Not Ready                 1907
Business Not Located        68
Name: count, dtype: int64

In [4]:
# Action 1: Quarantine rows without a trainable label
rows_before, cols_before = df_clean.shape

if 'Results' in df_clean.columns:
    results_clean = df_clean['Results'].astype('string').str.strip()
    quarantine_mask = results_clean.isna() | results_clean.eq('') | results_clean.isin(['Not Ready', 'Business Not Located', 'Out of Business', 'No Entry'])
    quarantine_df = df_clean.loc[quarantine_mask].copy()
    if not quarantine_df.empty:
        quarantine_df['quarantine_reason'] = 'missing_or_non_trainable_result'
    df_clean = df_clean.loc[~quarantine_mask].copy()
    df_clean['Results'] = results_clean.loc[~quarantine_mask]
else:
    quarantine_df = pd.DataFrame()

log_step('quarantine_non_trainable_labels', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Removed missing / Not Ready / Business Not Located labels from training set')
print('Quarantined rows:', len(quarantine_df))

Quarantined rows: 25206


In [5]:
# Final validation snapshot
post_profile = profile(df_clean, 'Post-clean profile')

print('Risk missing:', int(df_clean['Risk'].isna().sum()) if 'Risk' in df_clean.columns else 'N/A')
print('Risk levels:', df_clean['Risk'].value_counts(dropna=False).to_dict() if 'Risk' in df_clean.columns else 'N/A')
zip_values = df_clean['Zip'].astype('string').str.strip() if 'Zip' in df_clean.columns else pd.Series(dtype='string')
print('Malformed Zip count:', int(zip_values.dropna().str.match(r'^\d{5}$').eq(False).sum()) if 'Zip' in df_clean.columns else 'N/A')
print('Future LICENSE TERM START DATE count:', int(((df_clean['Inspection Date'].notna()) & (df_clean['LICENSE TERM START DATE'].notna()) & (df_clean['LICENSE TERM START DATE'] > df_clean['Inspection Date'])).sum()) if {'Inspection Date', 'LICENSE TERM START DATE'}.issubset(df_clean.columns) else 'N/A')
print('Violations recorded flag present:', 'violations_recorded' in df_clean.columns)
print('Has prior inspection flag present:', 'has_prior_inspection' in df_clean.columns)
print('License matched flag present:', 'license_matched' in df_clean.columns)
print('Duplicate Inspection ID:', int(df_clean.duplicated(subset=['Inspection ID']).sum()))

post_profile.head(20)

=== Post-clean profile ===
shape: (171470, 26)
full duplicates: 0
duplicate Inspection ID: 0
Risk missing: 0
Risk levels: {'High': 126239, 'Medium': 33117, 'Low': 12085, 'Unknown': 29}
Malformed Zip count: 171439
Future LICENSE TERM START DATE count: 0
Violations recorded flag present: True
Has prior inspection flag present: True
License matched flag present: True
Duplicate Inspection ID: 0


,missing_count,missing_pct,dtype
LICENSE TERM START DATE,147183,85.84,object
Violations,27541,16.06,object
COMMUNITY AREA NAME,10183,5.94,object
LICENSE TERM EXPIRATION DATE,8526,4.97,object
APPLICATION TYPE,8510,4.96,object
LICENSE DESCRIPTION,8510,4.96,object
LICENSE STATUS,8510,4.96,object
AKA Name,1767,1.03,object
Facility Type,659,0.38,object
Latitude,635,0.37,float64


In [6]:
# Save cleaned data, quarantine data, and logs
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
QUARANTINE_PATH.parent.mkdir(parents=True, exist_ok=True)
FINAL_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(PROCESSED_PATH, index=False)
df_clean.to_parquet(PROCESSED_PARQUET_PATH, index=False)
quarantine_df.to_csv(QUARANTINE_PATH, index=False)
save_log(LOG_PATH)
save_log(FINAL_LOG_PATH)

print('Saved cleaned data (csv) to:', PROCESSED_PATH)
print('Saved cleaned data (parquet) to:', PROCESSED_PARQUET_PATH)
print('Saved quarantine rows to:', QUARANTINE_PATH)
print('Saved cleaning log to:', LOG_PATH)
print('Saved final log copy to:', FINAL_LOG_PATH)

Saved cleaned data (csv) to: ..\..\data\processed\merged_inspections_licenses_inner_clean.csv
Saved cleaned data (parquet) to: ..\..\data\processed\merged_inspections_licenses_inner_clean.parquet
Saved quarantine rows to: ..\..\data\interim\cleaning\quarantine.csv
Saved cleaning log to: ..\..\data\interim\cleaning\cleaning_log.csv
Saved final log copy to: ..\..\data\interim\cleaning\cleaning_final_log.csv
